# Demo giữa kỳ — Hồi quy & Phân loại

**Cách chạy:** mở notebook này trong thư mục `ML/MidTerm/demo/`, chọn *Run All*. Toàn bộ chạy khoảng 10–20 giây, không cần mạng.
Lần đầu trên máy mới: `pip install -r ../requirements.txt`.

Notebook gọi **đúng code của bài** (`../src/models.py`, `../src/run_experiments.py`), không viết lại mô hình.

| Phần | Cho thầy thấy gì | Khớp với slide |
|---|---|---|
| 1 | Hồi quy tuyến tính tự cài bằng Gradient Descent | slide 3, 4, 6 |
| 2 | Kiểm tra gradient bằng sai phân trung tâm | slide 4 |
| 3 | Hồi quy logistic tự cài | slide 5, 7 |
| 4 | credit-g: pipeline chống rò rỉ, chạy lại logistic tự cài và MLP trên tập test | slide 8, 9, 11, 12 |
| 5 | Bảng 15 mô hình từ kết quả đã lưu | slide 10, 11 |

Nếu máy lỗi: kết quả đã được lưu sẵn bên dưới mỗi ô, và hình gốc nằm ở `../results/figures/`.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd().parent                      # thư mục ML/MidTerm
sys.path.insert(0, str(ROOT / 'src'))
from models import LinearRegressionGD, LogisticRegressionGD, ScratchLogisticClassifier, sigmoid, \
    linear_loss_gradient, logistic_loss_gradient
import run_experiments as rx                  # dùng lại load_credit, make_preprocessor
plt.rcdefaults()                              # bỏ font Times của report cho hình demo
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, f1_score, accuracy_score, roc_auc_score, confusion_matrix
SEED = 42
print('Sẵn sàng. Code mô hình lấy từ', ROOT / 'src' / 'models.py')

Sẵn sàng. Code mô hình lấy từ /home/user/TDTU-Coursework/ML/MidTerm/src/models.py


## 1. Hồi quy tuyến tính tự cài (slide 3, 4, 6)
Dữ liệu sinh từ công thức đã biết $y = 4 + 3x_1 - 2x_2 + 1.5x_3 + \varepsilon$, nên ta kiểm tra được mô hình có học đúng không.

In [2]:
rng = np.random.default_rng(SEED)
X = rng.uniform(-3, 3, (600, 3))
y = X @ np.array([3., -2., 1.5]) + 4. + rng.normal(0, 0.5, len(X))
train, test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=SEED)
lo, span = X[train].min(axis=0), np.ptp(X[train], axis=0)          # min–max học từ TRAIN
Xs = (X - lo) / span

lin = LinearRegressionGD(learning_rate=0.3, epochs=4000).fit(Xs[train], y[train])
pred = lin.predict(Xs[test])
w_raw = lin.w_ / span
print(f'Dừng sau {lin.n_iter_} vòng, chuẩn gradient = {lin.gradient_norm_:.1e}')
print('Trọng số học được:', np.round(w_raw, 2), ' (thật: 3, -2, 1.5)')
print(f'RMSE test = {np.sqrt(mean_squared_error(y[test], pred)):.4f}   R² = {r2_score(y[test], pred):.4f}')

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].semilogy(lin.loss_history_); ax[0].set(title='Loss giảm đơn điệu', xlabel='vòng GD', ylabel='MSE/2')
ax[1].scatter(y[test], pred, s=10); ax[1].plot([-12, 21], [-12, 21], 'k--')
ax[1].set(title='Dự đoán vs thật (test)', xlabel='y thật', ylabel='ŷ'); plt.tight_layout(); plt.show()

Dừng sau 1175 vòng, chuẩn gradient = 9.9e-09
Trọng số học được: [ 2.99 -2.01  1.49]  (thật: 3, -2, 1.5)
RMSE test = 0.5114   R² = 0.9947


## 2. Gradient có đúng không? (slide 4)
So gradient công thức với sai phân trung tâm $\frac{J(w+h e_j)-J(w-h e_j)}{2h}$ tại một điểm ngẫu nhiên.

In [3]:
def grad_check(loss_grad, X, y, h=1e-6):
    w, b = np.random.default_rng(0).normal(size=X.shape[1]), 0.3
    _, gw, gb = loss_grad(X, y, w, b)
    num = [(loss_grad(X, y, w + h*e, b)[0] - loss_grad(X, y, w - h*e, b)[0]) / (2*h) for e in np.eye(len(w))]
    num_b = (loss_grad(X, y, w, b + h)[0] - loss_grad(X, y, w, b - h)[0]) / (2*h)
    return max(np.abs(gw - num).max(), abs(gb - num_b))

yb = (y > np.median(y)).astype(int)
print(f'Linear   : sai khác lớn nhất = {grad_check(linear_loss_gradient, Xs[train], y[train]):.1e}')
print(f'Logistic : sai khác lớn nhất = {grad_check(logistic_loss_gradient, Xs[train], yb[train]):.1e}')
print('=> dưới 1e-6 nghĩa là đạo hàm viết đúng')

Linear   : sai khác lớn nhất = 1.4e-09
Logistic : sai khác lớn nhất = 9.5e-11
=> dưới 1e-6 nghĩa là đạo hàm viết đúng


## 3. Hồi quy logistic tự cài (slide 5, 7)
Sigmoid + cross-entropy, gradient $= \frac1n X^\top(p-y)$, ngưỡng 0.5.

In [4]:
rng = np.random.default_rng(SEED + 1)
X2 = rng.normal(size=(800, 2))
y2 = (rng.random(800) < sigmoid(0.8 + 1.6*X2[:, 0] - 1.2*X2[:, 1])).astype(int)
tr, te = train_test_split(np.arange(800), test_size=0.2, random_state=SEED, stratify=y2)
lo2, sp2 = X2[tr].min(axis=0), np.ptp(X2[tr], axis=0)
X2s = (X2 - lo2) / sp2

logm = LogisticRegressionGD(learning_rate=0.5, epochs=5000).fit(X2s[tr], y2[tr])
p = logm.predict_proba(X2s[te])[:, 1]; yhat = logm.predict(X2s[te])
print(f'Loss: {logm.loss_history_[0]:.4f} (= log 2) -> {logm.loss_history_[-1]:.4f} sau {logm.n_iter_} vòng')
print(f'Accuracy = {accuracy_score(y2[te], yhat):.4f}  F1 = {f1_score(y2[te], yhat):.4f}  AUC = {roc_auc_score(y2[te], p):.4f}')
print('Ma trận nhầm lẫn [[TN FP] [FN TP]]:', confusion_matrix(y2[te], yhat).tolist())

Loss: 0.6931 (= log 2) -> 0.4525 sau 5000 vòng
Accuracy = 0.8125  F1 = 0.8515  AUC = 0.8576
Ma trận nhầm lẫn [[TN FP] [FN TP]]: [[44, 18], [12, 86]]


## 4. credit-g: pipeline chống rò rỉ (slide 8, 9, 11, 12)
Chia test **trước**; min–max và one-hot nằm **trong pipeline**, nên chỉ học từ dữ liệu train.
Ở đây chạy lại 2 mô hình với tham số đã chọn bằng CV: logistic tự cài và MLP.

In [5]:
frame, audit = rx.load_credit()                        # kiểm MD5 file OpenML gốc
X3, y3 = frame.drop(columns='class'), (frame['class'] == 'bad').astype(int)
dev, tst = train_test_split(np.arange(len(y3)), test_size=.2, stratify=y3, random_state=SEED)
print('Số hồ sơ:', len(frame), '| good/bad:', audit['class_counts'], '| ô thiếu:', audit['missing_cells'])
print('Development / test:', len(dev), '/', len(tst))

Số hồ sơ: 1000 | good/bad: {'good': 700, 'bad': 300} | ô thiếu: 0
Development / test: 800 / 200


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

models = {
    'Logistic tự cài': ScratchLogisticClassifier(epochs=1500),
    'MLP (được chọn)': MLPClassifier(hidden_layer_sizes=(16,), activation='tanh', solver='sgd', alpha=0.,
                                     momentum=0., nesterovs_momentum=False, learning_rate_init=0.05,
                                     max_iter=500, tol=0., n_iter_no_change=501, random_state=SEED),
}
rows = {}
for name, model in models.items():
    pipe = Pipeline([('prep', rx.make_preprocessor(audit['categorical'])), ('model', model)])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')                  # MLP báo chưa hội tụ sau 500 epoch (đã ghi trong report)
        pipe.fit(X3.iloc[dev], y3.iloc[dev])             # fit CHỈ trên development
    pr = pipe.predict(X3.iloc[tst]); sc = pipe.predict_proba(X3.iloc[tst])[:, 1]
    cm = confusion_matrix(y3.iloc[tst], pr)
    rows[name] = {'accuracy': accuracy_score(y3.iloc[tst], pr), 'macro F1': f1_score(y3.iloc[tst], pr, average='macro'),
                  'AUC': roc_auc_score(y3.iloc[tst], sc), 'recall bad': cm[1, 1] / cm[1].sum(), 'bỏ sót bad': cm[1, 0]}
pd.DataFrame(rows).T.round(4)

,accuracy,macro F1,AUC,recall bad,bỏ sót bad
Logistic tự cài,0.78,0.7301,0.8011,0.5833,25.0
MLP (được chọn),0.79,0.7305,0.8133,0.5333,28.0


**Điểm cần nói:** accuracy 0.79 nghe tốt, nhưng MLP bỏ sót 28/60 hồ sơ xấu (recall bad 0.533). Logistic tự cài có macro F1 gần như bằng MLP.

## 5. Bảng đầy đủ 15 mô hình (slide 10, 11)
Đọc từ `results/q2_metrics.json`, tức kết quả của lần chạy `src/run_experiments.py` (chạy lại đủ 14 mô hình mất vài phút nên không chạy trực tiếp lúc demo).

In [7]:
q2 = json.load(open(ROOT / 'results' / 'q2_metrics.json'))
table = pd.DataFrame([{'mô hình': m['model'], 'CV macro F1': m['cv_macro_f1'], '± SD': m['cv_std'],
                       'test acc': m['accuracy'], 'test macro F1': m['macro_f1'], 'recall bad': m['recall_bad'],
                       'train−CV gap': m['gap']} for m in q2['models']])
table.sort_values('CV macro F1', ascending=False).round(3).reset_index(drop=True)

,mô hình,CV macro F1,± SD,test acc,test macro F1,recall bad,train−CV gap
0,MLP,0.696,0.064,0.790,0.730,0.533,0.064
1,Bernoulli NB,0.694,0.064,0.735,0.689,0.583,0.022
2,Polynomial SVM,0.686,0.062,0.775,0.713,0.517,0.225
3,Bagging,0.680,0.053,0.730,0.649,0.417,0.221
4,RBF SVM,0.669,0.066,0.745,0.695,0.567,0.321
5,Scratch logistic,0.666,0.074,0.780,0.730,0.583,0.065
6,Linear SVM,0.665,0.058,0.760,0.696,0.500,0.075
7,Gradient boosting,0.665,0.027,0.775,0.713,0.517,0.217
8,AdaBoost,0.664,0.056,0.785,0.726,0.533,0.072
9,Gaussian NB,0.662,0.058,0.710,0.677,0.650,0.046
